# W03B · 협업 필터링 계산을 끝까지 따라가기
**딥러닝응용I(추천시스템) · 동덕여자대학교 데이터사이언스전공 · 유원상 교수 · 2026-2**

2026-09-17. 기존 W03A를 주교안으로 사용하며 작은 표로 관측·유사도·영화별 집계를 다시 연결한다.
[보충 강의](https://github.com/lunalab-ai/recommender/blob/2026-fall-w03b/course/notion/sessions/w03b-cf-step-by-step.md)

CPU 런타임에서 첫 셀부터 실행한다. 합성 데이터는 자동 생성한다. `None`인 빈칸이 미완성이어도 독립 시연은 계속 실행할 수 있다. 예측을 적은 뒤 실행 결과와 비교한다.


## 1. 환경과 데이터
`sys.executable`은 현재 커널의 Python이다. 고정된 기존 수업 태그와 검증할 Gradio 버전을 설치한다. 설치 오류를 숨기지 않는다. 재시작 안내가 있으면 재시작 후 첫 셀부터 실행한다.

`toy_ratings()`는 인자 없이 18행의 `user_id/movie_id/rating` 표를 만든다. `UserCF(metric="cosine",min_common=3)`는 설정을 보관하고 `fit`이 고유한 관측 쌍으로 행렬·유사도·평균을 준비한다. `min_common`은 선택 Pearson에서 적용되며 이번 기본 코사인은 전체 영화 차원을 쓴다.
[toy_ratings 정의](https://github.com/lunalab-ai/recommender/blob/2026-fall-w03a/src/luna_recsys/collaborative.py#L15) · [UserCF 정의](https://github.com/lunalab-ai/recommender/blob/2026-fall-w03a/src/luna_recsys/collaborative.py#L44)


In [ ]:
import os, sys, subprocess
os.environ["GRADIO_ANALYTICS_ENABLED"]="False"
IN_COLAB = "google.colab" in sys.modules
COURSE_VERSION = "2026-fall-w03a"
if IN_COLAB:
    subprocess.run([sys.executable,"-m","pip","install","-q",
        f"luna-recommender-course[apps] @ git+https://github.com/lunalab-ai/recommender.git@{COURSE_VERSION}",
        "gradio==6.27.0"],check=True)
import numpy as np
import pandas as pd
import gradio as gr
from IPython.display import display
from luna_recsys.collaborative import UserCF, toy_ratings
from luna_recsys.cf_app import cf_view, build_cf_app
toy=toy_ratings()
model=UserCF().fit(toy)
R=model.rating_matrix_
movies=pd.DataFrame({"movie_id":[1,2,3,4,5],"title":list("ABCDE")})
display(R)
display(R.notna().astype(int))
display(R.fillna(0))
print("Python",sys.version.split()[0],"NumPy",np.__version__,"Gradio",gr.__version__)


### E1 · 관측과 0 채움
전체 관측 개수와 U1의 관측 평균을 구하라. 계산용 0을 평균에 넣은 결과와 다른 이유를 한 문장으로 적어라. `notna`는 관측 Boolean 표, `sum`은 True의 개수, `mean`은 기본적으로 NaN을 제외한 평균이다.


In [ ]:
observed_count=None
u1_observed_mean=None
if observed_count is None or u1_observed_mean is None:
    print("E1: 관측 개수와 관측 평균을 입력하세요.")
else:
    assert observed_count == R.notna().to_numpy().sum()
    assert np.isclose(u1_observed_mean,R.loc[1].mean())


## 2. 같은 열 순서에서 코사인 계산
두 길이 5 배열을 `np.dot`으로 곱하면 스칼라 내적이다. `np.linalg.norm`은 제곱합의 제곱근이다.
$$s(x,y)=\frac{x^\mathsf Ty}{\|x\|_2\|y\|_2}.$$
아래에는 내적과 각자의 제곱합을 표시한다. 공통 항목만 남기면 분모도 달라진다.


In [ ]:
x=R.loc[1].fillna(0).to_numpy()
y=R.loc[2].fillna(0).to_numpy()
display(pd.DataFrame({"U1":x,"U2":y,"곱":x*y,"U1 제곱":x*x,"U2 제곱":y*y},index=list("ABCDE")))
print("내적",np.dot(x,y),"제곱합",np.dot(x,x),np.dot(y,y))


### E2 · 두 노름의 곱
전체 차원 코사인 빈칸을 완성하라. 두 노름의 합으로 나누면 왜 코사인 정의가 아닌지도 설명하라.


In [ ]:
cosine_12=None
if cosine_12 is None:
    print("E2: 내적 / (첫 벡터 노름 × 둘째 벡터 노름)")
else:
    assert np.isclose(cosine_12,model.similarity_.loc[1,2])


## 3. 목표 영화별 근거 표
`predict_details(pairs)`는 ID 두 열을 입력받아 같은 행 순서로 예측·basis·기여자 수·정규화 전 유사도 합을 반환한다. `explain(u,i)`는 유사도 내림차순으로 실제 기여자의 평점, 정규화 `weight`, `contribution`을 반환한다. 빈 표는 평균 대체의 근거일 수 있다.
[predict_details 정의](https://github.com/lunalab-ai/recommender/blob/2026-fall-w03a/src/luna_recsys/collaborative.py#L133) · [explain 정의](https://github.com/lunalab-ai/recommender/blob/2026-fall-w03a/src/luna_recsys/collaborative.py#L172)


In [ ]:
pairs=pd.DataFrame({"user_id":[1,1,99,99],"movie_id":[4,5,4,99]})
details=model.predict_details(pairs)
display(pd.concat([pairs,details],axis=1))
for target in [4,5]:
    evidence=model.explain(1,target)
    display(evidence)
    print("목표",target,"정규화 전 합",evidence.similarity.sum(),
          "정규화 후 합",evidence.weight.sum(),"기여도 합",evidence.contribution.sum())


### E3 · 실행되지만 잘못된 분모
E를 예측하면서 모든 양의 유사도 사용자의 합을 분모로 쓰는 식을 고쳐라. 자신을 제외하고, E 관측이 있으며, 유사도가 양수인 **동일한** 마스크를 분자와 분모에 적용한다. `.loc[mask]`는 Boolean 조건으로 행을 고른다.


In [ ]:
s=model.similarity_.loc[1]
r=R[5]
wrong_denominator=s[(s.index!=1)&(s>0)].sum()
eligible=None
prediction_e=None
if eligible is None:
    print("E3: 관측 여부·자기 제외·양수 유사도를 결합하세요.")
else:
    prediction_e=float((s.loc[eligible]*r.loc[eligible]).sum()/s.loc[eligible].sum())
    print("수정한 E 예측:",prediction_e)


### E4 · 보류 정답이 학습에 남은 코드 수정
U2의 D 관측 한 개를 test로 보류한다. `UserCF().fit(toy)`로 학습한 뒤 그 관측을 평가하는 코드는 누수다. `training_for_fit`에 보류 쌍을 제외한 표를 넣고, 같은 사용자 ID가 train에 남는 것과 같은 관측 쌍이 남는 것을 구별해 설명하라.


In [ ]:
is_test=(toy.user_id==2)&(toy.movie_id==4)
test=toy.loc[is_test].copy()
training_for_fit=None
if training_for_fit is None:
    print("E4: test 관측 쌍을 제외한 train 표를 만드세요.")
else:
    audit_model=UserCF().fit(training_for_fit)
    print(audit_model.predict_details(test[["user_id","movie_id"]]))


## 4. 평점을 바꾸고 재학습하기
`copy`로 원래 표를 보존한다. U2의 D 평점을 5→1로 바꾼 뒤 새 모델을 만든다. 원래 모델은 이전 학습 상태를 유지하므로 결과를 나란히 비교할 수 있다.


In [ ]:
changed=toy.copy()
changed.loc[(changed.user_id==2)&(changed.movie_id==4),"rating"]=1.
changed_model=UserCF().fit(changed)
after=changed_model.predict_details(pairs.iloc[:2])
display(pd.DataFrame({"영화":["D","E"],"변경 전":details.prediction.iloc[:2].to_numpy(),"재학습 후":after.prediction.to_numpy()}))
print("U1-U2 유사도 전/후",model.similarity_.loc[1,2],changed_model.similarity_.loc[1,2])


### E5 · 결과와 원인을 구별하기
D·E 예측이 각각 어떻게 바뀌었는지 기록하라. E 열은 그대로인데 E 예측이 달라진 이유와, 평점만 바꾸고 기존 가중치를 고정한 계산이 다른 이유를 설명하라.


In [ ]:
explanation_e5=None
if explanation_e5 is None:
    print("E5: 실제 출력 숫자 → 유사도 분모 → 영화별 가중치 순서로 설명하세요.")


## 5. 공통 함수를 앱에 연결하기
`cf_view(model,movies,user_id,top_n=10)`은 서버 없이 호출할 수 있는 화면 콜백이다. 학습된 모델과 `movie_id/title` 카탈로그를 받아 요약 HTML, 미평가 영화 추천 표, 첫 추천 근거 표의 세 값을 반환한다. `top_n`은 목록 길이다. 모델을 다시 학습하지 않는다.
[cf_view 정의](https://github.com/lunalab-ai/recommender/blob/2026-fall-w03a/src/luna_recsys/cf_app.py#L11) · [build_cf_app 정의](https://github.com/lunalab-ai/recommender/blob/2026-fall-w03a/src/luna_recsys/cf_app.py#L39)

먼저 일반 함수로 입력과 출력을 확인한다. 이후 같은 함수를 버튼에 연결한다.


In [ ]:
summary,rows,reasons=cf_view(model,movies,1,2)
display(rows)
display(reasons)
print("출력 개수",len((summary,rows,reasons)))


### E6 · 자신의 버튼 이벤트 만들기
Gradio Blocks 안에서 사용자 Dropdown, 추천 수 Slider, Button을 만들고 출력 HTML·Dataframe·Dataframe 세 개를 만든다. `cf_view`를 호출하는 자신의 함수를 `button.click`에 연결하라. 입력 순서와 반환 순서를 설명하고 사용자 1과 5, 추천 수 1과 3으로 직접 조작하라. 원하면 근거 Accordion을 기본 펼침으로 바꾸어 보자.

`button.click(fn, inputs, outputs)`에서 입력 컴포넌트 수·순서는 fn의 인자와 일치해야 하고 출력 수·순서는 반환값과 일치해야 한다. 아래 기본 앱 시연은 미완성 문제와 독립적으로 실행할 수 있다.


In [ ]:
my_app=None
if my_app is None:
    print("E6: 이 셀에 자신의 UI와 이벤트 연결을 작성하세요.")


### 누적 앱 실행
`build_cf_app`은 실행 전 앱 객체를 반환한다. Colab에서는 아래 셀이 공유 링크를 만든다. 런타임이 종료되면 링크도 사용할 수 없으므로 다음 수업에는 이 notebook부터 다시 실행한다. 자신의 앱이 완성되면 같은 방식으로 `my_app.launch(share=True)`를 호출할 수 있다.


In [ ]:
demo=build_cf_app(model,movies)
if IN_COLAB:
    demo.launch(share=True,prevent_thread_lock=True)
else:
    print("로컬 검증: 앱 객체와 콜백을 검사합니다. 브라우저 시연은 별도로 launch합니다.")


## 실행 결과 확인
공개 시연의 수치·근거 합·평가 후보 제외·앱 콜백을 확인한다. 아래 성공은 현재 커널의 검사이며 다른 환경의 실행을 대신하지 않는다.


In [ ]:
assert R.shape==(5,5) and int(R.notna().sum().sum())==18
assert np.isclose(details.iloc[0].prediction,3.685360466886743)
for movie_id in [4,5]:
    ev=model.explain(1,movie_id)
    p=model.predict_details(pd.DataFrame({"user_id":[1],"movie_id":[movie_id]})).iloc[0]
    assert np.isclose(ev.weight.sum(),1)
    assert np.isclose(ev.similarity.sum(),p.weight_sum)
    assert np.isclose(ev.contribution.sum(),p.prediction)
    assert ev.rating.min()<=p.prediction<=ev.rating.max()
assert details.iloc[2].basis=="movie-mean" and details.iloc[3].basis=="global-mean"
for uid in [1,5]:
    for n in [1,3]:
        h,rs,es=cf_view(model,movies,uid,n)
        assert h and 0<len(rs)<=n
        seen=set(toy.loc[toy.user_id==uid,"movie_id"])
        assert not seen & set(rs.movie_id)
print("W03B runtime checks passed")


## 점검 퀴즈
1. 원본 평점표와 코사인 계산용 0 채움 표를 별도로 유지해야 하는 이유는 무엇인가?
2. U1–U2의 전체 차원 코사인에서 두 번째 노름에 D와 E 평점도 포함되는 이유는 무엇인가?
3. U1의 D와 E 예측에서 평가자 집합과 가중평균 분모가 달라지는 이유는 무엇인가?
4. predict_details의 weight_sum과 explain의 weight 합은 각각 무엇인가?
5. 학습에 없던 사용자에게 숫자 예측이 반환되면 개인화 근거가 충분하다고 볼 수 있는가?


## 정리와 참고자료
원본 관측 마스크, 코사인의 두 노름, 영화별 평가자 집합, 정규화 전·후 가중치 합, 평균 대체를 구별한다. 추천 목록 길이는 이웃 수가 아니다. 평가 정답은 fit에 넣지 않는다.

- 임일(2025), 주교재 3.1–3.3.
- [기존 수업 API 안내](https://github.com/lunalab-ai/recommender/blob/2026-fall-w03a/src/W03A-API.md)
- [scikit-learn cosine_similarity](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html)
- [Gradio 공식 배포 설명](https://pypi.org/project/gradio/6.27.0/)
